# Course 1: From Beginner to an AI Research Lab
## Session 3: Code like a researcher

**Goal:** turn the session 2 code into a small reproducible repository: a config, a fixed seed, one logged row per run, and a README.

**How to use this notebook:** run the cells from top to bottom. Cells marked `TODO` are for you to complete. The lab guide, quiz and assignment for this session are in the course document.

**Testing note:** every cell without a NOT RUN label was run once on a CPU (November 2026). Cells labelled NOT RUN need a GPU, a download or an account. Cells that are worksheets hold example rows for you to replace with your own.

### 1. The layout
```
my-project/
  README.md
  requirements.txt
  configs/base.json      {"lr": 0.1, "steps": 200, "seed": 0}
  src/model.py           loss and grad functions
  train.py               reads a config, trains, logs one row
  scripts/plot.py        reads results/, saves the figure
  results/runs.csv
```
This notebook writes that repository to disk so you can see every file. In real work you create it by hand and commit as you go.

In [ ]:
import os, json, subprocess, sys, textwrap
ROOT = "my-project"
for d in ["configs", "src", "scripts", "results"]: os.makedirs(os.path.join(ROOT, d), exist_ok=True)
def write(path, text):
    with open(os.path.join(ROOT, path), "w") as f: f.write(textwrap.dedent(text).lstrip("\n"))

write("configs/base.json", json.dumps({"lr": 0.1, "steps": 200, "seed": 0}, indent=1))
write("src/__init__.py", "")
write("src/model.py", """
    import numpy as np

    def make_data(seed, n=200):
        rng = np.random.default_rng(seed)          # the seed comes from the config, never from a hidden default
        x = rng.uniform(-2, 2, n)
        return x, 3 * x + 2 + rng.normal(0, 0.5, n)

    def loss(w, b, x, y):
        return float(np.mean((w * x + b - y) ** 2))

    def grad(w, b, x, y):
        r = w * x + b - y
        return 2 * np.mean(r * x), 2 * np.mean(r)
""")
write("src/utils.py", """
    import random, subprocess
    import numpy as np

    def set_seed(seed):
        random.seed(seed)
        np.random.seed(seed)
        try:
            import torch
            torch.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
        except ImportError:
            pass

    def git_hash():
        try:
            out = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], stderr=subprocess.DEVNULL)
            return out.decode().strip()
        except Exception:
            return "no-git"
""")
write("train.py", """
    import argparse, csv, json, os, time
    from src.model import make_data, loss, grad
    from src.utils import set_seed, git_hash

    def main():
        ap = argparse.ArgumentParser(); ap.add_argument("--config", required=True); args = ap.parse_args()
        cfg = json.load(open(args.config))
        set_seed(cfg["seed"])                      # first thing
        x, y = make_data(cfg["seed"])
        t0 = time.time(); w, b = 0.0, 0.0
        for _ in range(cfg["steps"]):
            gw, gb = grad(w, b, x, y)
            w, b = w - cfg["lr"] * gw, b - cfg["lr"] * gb
        row = {"time": time.strftime("%Y-%m-%d %H:%M:%S"), "commit": git_hash(), **cfg,
               "final_loss": round(loss(w, b, x, y), 10), "runtime_s": round(time.time() - t0, 4)}
        os.makedirs("results", exist_ok=True); path = "results/runs.csv"; new = not os.path.exists(path)
        with open(path, "a", newline="") as f:
            wr = csv.DictWriter(f, fieldnames=list(row)); new and wr.writeheader(); wr.writerow(row)
        print(row)

    if __name__ == "__main__":
        main()
""")
write("scripts/plot.py", """
    import pandas as pd, matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    df = pd.read_csv("results/runs.csv")
    df.plot.scatter(x="seed", y="final_loss"); plt.savefig("results/final_loss_by_seed.png")
    print("saved results/final_loss_by_seed.png from", len(df), "runs")
""")
write("requirements.txt", "numpy\npandas\nmatplotlib\n")
write("README.md", """
    # my-project
    Fits a line with gradient descent (course 6, session 3).

    Install: `pip install -r requirements.txt`
    Reproduce the figure: `python train.py --config configs/base.json && python scripts/plot.py`
""")
if os.path.exists(os.path.join(ROOT, "results", "runs.csv")): os.remove(os.path.join(ROOT, "results", "runs.csv"))
print(sorted(os.path.join(dp, f).replace(ROOT + os.sep, "") for dp, _, fs in os.walk(ROOT) for f in fs if "__pycache__" not in dp))

### 2. One command runs one experiment
Run it twice with the same config, then once with another seed.

In [ ]:
def run(config): return subprocess.run([sys.executable, "train.py", "--config", config], cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip()
print(run("configs/base.json")); print(run("configs/base.json"))
json.dump({"lr": 0.1, "steps": 200, "seed": 1}, open(os.path.join(ROOT, "configs", "seed1.json"), "w"))
print(run("configs/seed1.json"))

### 3. The reproducibility test
Same config, same result. Different seed, different result.

In [ ]:
import pandas as pd
runs = pd.read_csv(os.path.join(ROOT, "results", "runs.csv")); print(runs.to_string(index=False))
same = runs.final_loss[0] == runs.final_loss[1]; different = runs.final_loss[0] != runs.final_loss[2]
print("two runs with the same config agree exactly:", same, "| a different seed gives a different loss:", different)
assert same and different
print(subprocess.run([sys.executable, "scripts/plot.py"], cwd=ROOT, capture_output=True, text=True).stdout.strip())

A trap from session 2: that code called `np.random.default_rng(0)` directly. A generator created that way ignores `np.random.seed`, so `set_seed` alone would not have changed the data. Here the config's seed is passed into `make_data`. Know where every random number in your project comes from.

### 4. Version control (NOT RUN here)
Run these in a terminal inside the project folder. The `commit` column in `runs.csv` shows `no-git` until you do.

In [ ]:
# NOT RUN: needs git configured with your name and email, and a GitHub account for the last two lines.
#   git init
#   printf "__pycache__/\n*.png\n" > .gitignore
#   git add . && git commit -m "Line fit: config, seed, logged runs"
#   python train.py --config configs/base.json        # the commit column now holds a hash
#   git remote add origin https://github.com/<you>/my-project.git
#   git push -u origin main
# Rule: commit before you run. A result whose code was never committed cannot be traced back to the code that made it.

### 5. The clone test
Ask a classmate to clone the repository and run the README's one command with no help. Every question they ask is a gap in the README.

### TODO
1. Create this repository by hand, commit, and push it.
2. Add a `--seed` override on the command line and log it.
3. Pin exact versions in `requirements.txt` with `pip freeze`, and note your Python version in the README.